# Titan Experiment Runner
This notebook sets up the environment and runs the Associative Recall experiment on the Titans model.

## 1. Install Dependencies

In [ ]:
# Install dependencies
!pip install torch deepspeed transformers tqdm einops sentence-transformers

## 2. Setup File System

In [ ]:
import os
import base64

print(f'Current working directory: {os.getcwd()}')

def write_file(path, b64_content):
    target_path = os.path.abspath(path)
    directory = os.path.dirname(target_path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(target_path, "wb") as f:
        f.write(base64.b64decode(b64_content))
    print(f"Created {target_path}")

files = {
    'src/main.py': '...',
    'projects/original_benchmarks/associative_recall.py': 'CmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKaW1wb3J0IHJhbmRvbQoKY2xhc3MgQXNzb2NpYXRpdmVSZWNhbGxEYXRhc2V0KERhdGFzZXQpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHZvY2FiX3NpemU9MTAwLCBzZXFfbGVuPTEyOCwgbnVtX3NhbXBsZXM9MTAwMDAsIHBhaXJfbGVuPTIpOgogICAgICAgICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHZvY2FiX3NpemU6IFNpemUgb2YgdGhlIHZvY2FidWxhcnkgKGtleXMgYW5kIHZhbHVlcykuCiAgICAgICAgICAgIHNlcV9sZW46IFRvdGFsIGxlbmd0aCBvZiB0aGUgc2VxdWVuY2UuCiAgICAgICAgICAgIG51bV9zYW1wbGVzOiBOdW1iZXIgb2Ygc2FtcGxlcyBpbiB0aGUgZGF0YXNldC4KICAgICAgICAgICAgcGFpcl9sZW46IE51bWJlciBvZiB0b2tlbnMgcGVyIGtleS12YWx1ZSBwYWlyICh1c3VhbGx5IDI6IGtleSwgdmFsdWUpLgogICAgICAgICIiIgogICAgICAgIHNlbGYudm9jYWJfc2l6ZSA9IHZvY2FiX3NpemUKICAgICAgICBzZWxmLnNlcV9sZW4gPSBzZXFfbGVuCiAgICAgICAgc2VsZi5udW1fc2FtcGxlcyA9IG51bV9zYW1wbGVzCiAgICAgICAgc2VsZi5wYWlyX2xlbiA9IHBhaXJfbGVuCiAgICAgICAgCiAgICAgICAgIyBSZXNlcnZlIGV4cGxpY2l0IHRva2VucwogICAgICAgIHNlbGYuUVVFUllfVE9LRU4gPSB2b2NhYl9zaXplIC0gMSAgIyBOb3Qgc3RyaWN0bHkgbmVlZGVkIGlmIHBvc2l0aW9uIGltcGxpY2l0bHkgZGVmaW5lcyBxdWVyeSwgYnV0IGdvb2QgZm9yIGNsYXJpdHkgaWYgd2UgdXNlZCBzcGVjaWFsIHRva2VucwogICAgICAgICMgRm9yIHNpbXBsaWNpdHkgaW4gdGhpcyBudW1lcmljIHRhc2ssIHdlIGp1c3QgdXNlIHRoZSBudW1iZXJzIFswLCB2b2NhYl9zaXplLTJdIGZvciBkYXRhCiAgICAgICAgCiAgICBkZWYgX19sZW5fXyhzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5udW1fc2FtcGxlcwoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpZHgpOgogICAgICAgICMgMS4gR2VuZXJhdGUgdW5pcXVlIGtleXMKICAgICAgICAjIFdlIG5lZWQgZW5vdWdoIGtleXMgdG8gZmlsbCBtb3N0IG9mIHRoZSBzZXF1ZW5jZS4KICAgICAgICAjIE1heCBwYWlycyBhcHByb3ggc2VxX2xlbiAvIDIKICAgICAgICBudW1fcGFpcnMgPSAoc2VsZi5zZXFfbGVuIC0gMSkgLy8gMiAKICAgICAgICAKICAgICAgICAjIEVuc3VyZSB2b2NhYiBzaXplIGlzIHN1ZmZpY2llbnQgZm9yIHVuaXF1ZSBrZXlzIGlmIHBvc3NpYmxlLCBvciBhbGxvdyByZXBlYXRzIGlmIHZvY2FiIGlzIHNtYWxsIChidXQgQVIgdXN1YWxseSByZXF1aXJlcyB1bmlxdWUga2V5cyBmb3IgcXVlcnlpbmcpCiAgICAgICAgIyBMZXQncyBhc3N1bWUgdm9jYWJfc2l6ZSA+PiBudW1fcGFpcnMgdXN1YWxseSwgb3Igd2Ugc2FtcGxlIHdpdGggcmVwbGFjZW1lbnQgYnV0IHRoYXQgbWFrZXMgdGFzayBoYXJkZXIvYW1iaWd1b3VzLgogICAgICAgICMgU3RhbmRhcmQgQVI6IGtleXMgYXJlIHVuaXF1ZS4KICAgICAgICAKICAgICAgICBhdmFpbGFibGVfa2V5cyA9IGxpc3QocmFuZ2Uoc2VsZi52b2NhYl9zaXplIC0gMTApKSAjIFJlc2VydmUgc29tZSB0b3AgdG9rZW5zIGZvciBzcGVjaWFsIHVzZSBpZiBuZWVkZWQKICAgICAgICBpZiBsZW4oYXZhaWxhYmxlX2tleXMpIDwgbnVtX3BhaXJzOgogICAgICAgICAgICAjIElmIHZvY2FiIGlzIHNtYWxsLCB3ZSByZWR1Y2UgbnVtX3BhaXJzCiAgICAgICAgICAgIG51bV9wYWlycyA9IGxlbihhdmFpbGFibGVfa2V5cykKICAgICAgICAgICAgCiAgICAgICAga2V5cyA9IHJhbmRvbS5zYW1wbGUoYXZhaWxhYmxlX2tleXMsIG51bV9wYWlycykKICAgICAgICB2YWx1ZXMgPSBbcmFuZG9tLmNob2ljZShhdmFpbGFibGVfa2V5cykgZm9yIF8gaW4gcmFuZ2UobnVtX3BhaXJzKV0KICAgICAgICAKICAgICAgICAjIDIuIFNlbGVjdCBhIHRhcmdldCBxdWVyeQogICAgICAgIHRhcmdldF9pZHggPSByYW5kb20ucmFuZGludCgwLCBudW1fcGFpcnMgLSAxKQogICAgICAgIHF1ZXJ5X2tleSA9IGtleXNbdGFyZ2V0X2lkeF0KICAgICAgICB0YXJnZXRfdmFsID0gdmFsdWVzW3RhcmdldF9pZHhdCiAgICAgICAgCiAgICAgICAgIyAzLiBDb25zdHJ1Y3Qgc2VxdWVuY2UKICAgICAgICAjIGlucHV0OiBrMSB2MSBrMiB2MiAuLi4ga04gdk4gcXVlcnlfa2V5CiAgICAgICAgIyBsYWJlbDogLTEgLTEgLTEgLTEgLi4uIC0xIC0xIHRhcmdldF92YWwKICAgICAgICAKICAgICAgICBpbnB1dF9pZHMgPSBbXQogICAgICAgIGxhYmVscyA9IFtdCiAgICAgICAgCiAgICAgICAgZm9yIGssIHYgaW4gemlwKGtleXMsIHZhbHVlcyk6CiAgICAgICAgICAgIGlucHV0X2lkcy5leHRlbmQoW2ssIHZdKQogICAgICAgICAgICBsYWJlbHMuZXh0ZW5kKFstMTAwLCAtMTAwXSkgIyBJZ25vcmUgbG9zcyBmb3IgY29udGV4dAogICAgICAgICAgICAKICAgICAgICAjIEFwcGVuZCBxdWVyeQogICAgICAgIGlucHV0X2lkcy5hcHBlbmQocXVlcnlfa2V5KQogICAgICAgIGxhYmVscy5hcHBlbmQodGFyZ2V0X3ZhbCkKICAgICAgICAKICAgICAgICAjIFBhZGRpbmcgaWYgbmVlZGVkIChzaW1wbGlmaWVkOiBqdXN0IHRydW5jYXRlIG9yIGFzc3VtZSBmaXhlZCBzaXplIGxvZ2ljIG1hdGNoZXMpCiAgICAgICAgIyBGb3IgdGhpcyBkZW1vLCB3ZSByZXR1cm4gdGVuc29ycyBkaXJlY3RseS4KICAgICAgICAjIElmIHRoZSBnZW5lcmF0ZWQgbGVuZ3RoIDwgc2VsZi5zZXFfbGVuLCB3ZSBtaWdodCBwYWQgKGJ1dCBiYXRjaGluZyBoYW5kbGVzIGl0IGlmIHdlIHVzZSBjb2xsYXRlKS4KICAgICAgICAjIExldCdzIHNpbXBsZSByZXR1cm4gYXMgaXMsIGFuZCB0aGUgbW9kZWwgaGFuZGxlcyB2YXJpYWJsZSBsZW5ndGggb3Igd2UgbWFrZSBpdCBmaXhlZC4KICAgICAgICAKICAgICAgICByZXR1cm4gdG9yY2gudGVuc29yKGlucHV0X2lkcywgZHR5cGU9dG9yY2gubG9uZyksIHRvcmNoLnRlbnNvcihsYWJlbHMsIGR0eXBlPXRvcmNoLmxvbmcpCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgZHMgPSBBc3NvY2lhdGl2ZVJlY2FsbERhdGFzZXQoKQogICAgeCwgeSA9IGRzWzBdCiAgICBwcmludCgiSW5wdXQ6IiwgeCkKICAgIHByaW50KCJMYWJlbDoiLCB5KQo=',
    'projects/original_benchmarks/train_toy.py': 'CmltcG9ydCBzeXMKaW1wb3J0IG9zCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IGFyZ3BhcnNlCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlcgpmcm9tIHRxZG0gaW1wb3J0IHRxZG0KCiMgQWRkIHBhcmVudCBkaXJlY3RvcnkgdG8gcGF0aCB0byBpbXBvcnQgbWFpbiAob3JpZ2luYWxfYmVuY2htYXJrcyAtPiBwcm9qZWN0cyAtPiBUaXRhbkxMTSkKc3lzLnBhdGguYXBwZW5kKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSkpCnN5cy5wYXRoLmFwcGVuZChvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpKSwgJ3NyYycpKQpmcm9tIG1haW4gaW1wb3J0IFRpdGFuTUFDLCBUaXRhbk1BRywgVGl0YW5NQUwsIFRpdGFuTW9kZWxGb3JMTQoKZnJvbSBhc3NvY2lhdGl2ZV9yZWNhbGwgaW1wb3J0IEFzc29jaWF0aXZlUmVjYWxsRGF0YXNldAoKZGVmIHRyYWluKGFyZ3MpOgogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBwcmludChmIlVzaW5nIGRldmljZToge2RldmljZX0iKQoKICAgICMgRGF0YXNldAogICAgdHJhaW5fZHMgPSBBc3NvY2lhdGl2ZVJlY2FsbERhdGFzZXQodm9jYWJfc2l6ZT1hcmdzLnZvY2FiX3NpemUsIHNlcV9sZW49YXJncy5zZXFfbGVuLCBudW1fc2FtcGxlcz0xMDAwMCkKICAgIHRyYWluX2RsID0gRGF0YUxvYWRlcih0cmFpbl9kcywgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoX3NpemUsIHNodWZmbGU9VHJ1ZSkKICAgIAogICAgIyBNb2RlbCBTZXR1cAogICAgaWYgYXJncy5tb2RlbF92YXJpYW50ID09ICJNQUMiOgogICAgICAgIHRpdGFuX21vZHVsZSA9IFRpdGFuTUFDKGFyZ3MuZGltLCBhcmdzLmNodW5rX3NpemUsIGFyZ3MuaGlkZGVuX2RpbSwgYXJncy5tZW1vcnlfZGVwdGgsIGFyZ3MubnVtX3BlcnNpc3RlbnRfdG9rZW5zLCB0aHJlc2hvbGQ9YXJncy50aHJlc2hvbGQpCiAgICBlbGlmIGFyZ3MubW9kZWxfdmFyaWFudCA9PSAiTUFHIjoKICAgICAgICB0aXRhbl9tb2R1bGUgPSBUaXRhbk1BRyhhcmdzLmRpbSwgYXJncy5oaWRkZW5fZGltLCBhcmdzLm1lbW9yeV9kZXB0aCwgYXJncy5udW1fcGVyc2lzdGVudF90b2tlbnMsIGFyZ3Mud2luZG93X3NpemUsIHRocmVzaG9sZD1hcmdzLnRocmVzaG9sZCkKICAgIGVsc2U6ICMgTUFMCiAgICAgICAgdGl0YW5fbW9kdWxlID0gVGl0YW5NQUwoYXJncy5kaW0sIGFyZ3MuaGlkZGVuX2RpbSwgYXJncy5tZW1vcnlfZGVwdGgsIGFyZ3MubnVtX3BlcnNpc3RlbnRfdG9rZW5zLCBhcmdzLndpbmRvd19zaXplLCB0aHJlc2hvbGQ9YXJncy50aHJlc2hvbGQpCiAgICAgICAgCiAgICBtb2RlbCA9IFRpdGFuTW9kZWxGb3JMTSh0aXRhbl9tb2R1bGUsIGFyZ3Mudm9jYWJfc2l6ZSwgYXJncy5kaW0pCiAgICBtb2RlbC50byhkZXZpY2UpCiAgICAKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW1XKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9MWUtNCkKICAgIAogICAgbW9kZWwudHJhaW4oKQogICAgCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoYXJncy5udW1fZXBvY2hzKToKICAgICAgICB0b3RhbF9sb3NzID0gMAogICAgICAgIGNvcnJlY3QgPSAwCiAgICAgICAgdG90YWwgPSAwCiAgICAgICAgCiAgICAgICAgcGJhciA9IHRxZG0odHJhaW5fZGwsIGRlc2M9ZiJFcG9jaCB7ZXBvY2grMX0iKQogICAgICAgIGZvciBpbnB1dF9pZHMsIGxhYmVscyBpbiBwYmFyOgogICAgICAgICAgICBpbnB1dF9pZHMsIGxhYmVscyA9IGlucHV0X2lkcy50byhkZXZpY2UpLCBsYWJlbHMudG8oZGV2aWNlKQogICAgICAgICAgICAKICAgICAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZCgpCiAgICAgICAgICAgIGxvZ2l0cywgbG9zcyA9IG1vZGVsKGlucHV0X2lkcywgbGFiZWxzKSAjIFRpdGFuTW9kZWxGb3JMTSBoYW5kbGVzIGxvc3MgY2FsY3VsYXRpb24gaWYgbGFiZWxzIHByb3ZpZGVkCiAgICAgICAgICAgIAogICAgICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgICAgICAKICAgICAgICAgICAgdG90YWxfbG9zcyArPSBsb3NzLml0ZW0oKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBBY2N1cmFjeSBNZXRyaWMKICAgICAgICAgICAgIyBsYWJlbHMgaGF2ZSAtMTAwIGZvciBpZ25vcmVkIHBvc2l0aW9ucy4gV2Ugb25seSBjYXJlIGFib3V0IHRoZSBsYXN0IHRva2VuIHVzdWFsbHkuCiAgICAgICAgICAgICMgRmxhdHRlbiBmb3IgYXJnbWF4CiAgICAgICAgICAgIHByZWRfdG9rZW5zID0gdG9yY2guYXJnbWF4KGxvZ2l0cywgZGltPS0xKQogICAgICAgICAgICBtYXNrID0gbGFiZWxzICE9IC0xMDAKICAgICAgICAgICAgCiAgICAgICAgICAgIGNvcnJlY3QgKz0gKHByZWRfdG9rZW5zW21hc2tdID09IGxhYmVsc1ttYXNrXSkuc3VtKCkuaXRlbSgpCiAgICAgICAgICAgIHRvdGFsICs9IG1hc2suc3VtKCkuaXRlbSgpCiAgICAgICAgICAgIAogICAgICAgICAgICBwYmFyLnNldF9wb3N0Zml4KHsiTG9zcyI6IGYie2xvc3MuaXRlbSgpOi40Zn0iLCAiQWNjIjogZiJ7Y29ycmVjdC90b3RhbDouMiV9In0pCiAgICAgICAgICAgIAogICAgcHJpbnQoIlRyYWluaW5nIGZpbmlzaGVkLiIpCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tb2RlbF92YXJpYW50IiwgdHlwZT1zdHIsIGRlZmF1bHQ9Ik1BQyIsIGNob2ljZXM9WyJNQUMiLCAiTUFHIiwgIk1BTCJdKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS12b2NhYl9zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTI4KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZXFfbGVuIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NjQpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRpbSIsIHR5cGU9aW50LCBkZWZhdWx0PTEyOCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0taGlkZGVuX2RpbSIsIHR5cGU9aW50LCBkZWZhdWx0PTEyOCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbWVtb3J5X2RlcHRoIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbnVtX3BlcnNpc3RlbnRfdG9rZW5zIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY2h1bmtfc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTMyKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13aW5kb3dfc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTMyKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1iYXRjaF9zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTYpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW51bV9lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD01KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10aHJlc2hvbGQiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMCkKICAgIAogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHRyYWluKGFyZ3MpCg==',
    'projects/hybrid_titans/rag_compare.py': 'aW1wb3J0IHRvcmNoCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgcmFuZG9tCmltcG9ydCBzeXMKaW1wb3J0IG9zCgojIEFkZCBwYXJlbnQgZGlyZWN0b3J5IHRvIHBhdGggdG8gaW1wb3J0IG1haW4gKGh5YnJpZF90aXRhbnMgLT4gcHJvamVjdHMgLT4gVGl0YW5MTE0pCnN5cy5wYXRoLmFwcGVuZChvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkpKQpzeXMucGF0aC5hcHBlbmQob3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSksICdzcmMnKSkKCmZyb20gbWFpbiBpbXBvcnQgVGl0YW5NQUMsIFRpdGFuTUFHLCBUaXRhbk1BTCwgVGl0YW5SQUcKCmRlZiBnZW5lcmF0ZV9oYXlzdGFjayhudW1fZG9jcywgZG9jX2xlbiwgZGltLCBrZXlfaWR4PU5vbmUpOgogICAgIiIiCiAgICBHZW5lcmF0ZXMgTiBkb2N1bWVudHMuIE9uZSBjb250YWluaW5nIHRoZSAnbmVlZGxlJyAocGFzc2tleSksIG90aGVycyBub2lzZS4KICAgICIiIgogICAgZG9jcyA9IFtdCiAgICBuZWVkbGVfcGF0dGVybiA9IHRvcmNoLnJhbmRuKDEsIGRpbSkKICAgIAogICAgaWYga2V5X2lkeCBpcyBOb25lOgogICAgICAgIGtleV9pZHggPSByYW5kb20ucmFuZGludCgwLCBudW1fZG9jcyAtIDEpCiAgICAgICAgCiAgICBmb3IgaSBpbiByYW5nZShudW1fZG9jcyk6CiAgICAgICAgZG9jID0gdG9yY2gucmFuZG4oZG9jX2xlbiwgZGltKQogICAgICAgIGlmIGkgPT0ga2V5X2lkeDoKICAgICAgICAgICAgZG9jWy0xLCA6XSA9IG5lZWRsZV9wYXR0ZXJuCiAgICAgICAgZG9jcy5hcHBlbmQoZG9jKQogICAgICAgIAogICAgcmV0dXJuIGRvY3MsIG5lZWRsZV9wYXR0ZXJuLCBrZXlfaWR4CgpkZWYgcHJldHJhaW5faWRlbnRpdHkobW9kZWwsIGRpbSwgc3RlcHM9NTAwKToKICAgICIiIgogICAgVHJhaW5zIHRoZSBtb2RlbCdzIEF0dGVudGlvbi9Qcm9jZXNzaW5nIGxheWVycyB0byBwZXJmb3JtIElkZW50aXR5IE1hcHBpbmcgZih4KT14LgogICAgIiIiCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtVyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPTAuMDEpCiAgICAKICAgIHByaW50KGYiICBbUHJlLXRyYWluaW5nXSBUZWFjaGluZyBtb2RlbCB0byBjb3B5IChJZGVudGl0eSBGdW5jdGlvbikuLi4iKQogICAgCiAgICAjIFtGaXhdIERpc2FibGUgVGl0YW4ncyBpbnRlcm5hbCBUZXN0LVRpbWUgVHJhaW5pbmcgZHVyaW5nIElkZW50aXR5IFByZS10cmFpbmluZy4KICAgICMgV2Ugb25seSB3YW50IHRvIHRyYWluIHRoZSBBdHRlbnRpb24gbWVjaGFuaXNtIChXZWlnaHRzKSwgbm90IHRoZSBNZW1vcnkgKFN0YXRlKS4KICAgICMgT3RoZXJ3aXNlLCBpbi1wbGFjZSB1cGRhdGVzIGNhdXNlICdiYWNrd2FyZCB0aHJvdWdoIGdyYXBoIHNlY29uZCB0aW1lJyBlcnJvci4KICAgIHNhdmVkX3RocmVzaG9sZCA9IG1vZGVsLnRocmVzaG9sZAogICAgbW9kZWwudGhyZXNob2xkID0gZmxvYXQoJ2luZicpICMgRm9yY2Ugc2tpcCB1cGRhdGUKICAgIAogICAgZm9yIGkgaW4gcmFuZ2Uoc3RlcHMpOgogICAgICAgICMgR2VuZXJhdGUgcmFuZG9tIGRhdGEKICAgICAgICB4ID0gdG9yY2gucmFuZG4oMSwgMzIsIGRpbSkgIyBbQmF0Y2gsIFNlcSwgRGltXQogICAgICAgIHRhcmdldCA9IHgKICAgICAgICAKICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICAKICAgICAgICBwcmVkID0gbW9kZWwoeCkKICAgICAgICBsb3NzID0gKChwcmVkIC0gdGFyZ2V0KSoqMikubWVhbigpCiAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgIAogICAgICAgIGlmIGkgJSAyMCA9PSAwOgogICAgICAgICAgICBwcmludChmIiAgICBTdGVwIHtpfTogTG9zcyB7bG9zcy5pdGVtKCk6LjRmfSIpCiAgICAgICAgICAgIAogICAgIyBSZXN0b3JlIG9yaWdpbmFsIHRocmVzaG9sZCBmb3IgRXhwZXJpbWVudAogICAgbW9kZWwudGhyZXNob2xkID0gc2F2ZWRfdGhyZXNob2xkCgpkZWYgcnVuX3NpbmdsZV92YXJpYW50KHZhcmlhbnRfbmFtZSwgTW9kZWxDbGFzcywgYXJncywgZG9jc19iYXRjaCwga2V5X2lkeCk6CiAgICBwcmludChmIlxuW3t2YXJpYW50X25hbWV9XSBUZXN0aW5nIE1lbW9yeSBSZXRlbnRpb24uLi4iKQogICAgCiAgICAjIDEuIENvbmZpZyBIYW5kbGluZwogICAgIyBNQUMgdXNlcyAnY2h1bmtfc2l6ZScsIE1BRy9NQUwgdXNlICd3aW5kb3dfc2l6ZScKICAgIGNvbmZpZyA9IHsKICAgICAgICAiZGltIjogYXJncy5kaW0sCiAgICAgICAgImhpZGRlbl9kaW0iOiBhcmdzLmRpbSwKICAgICAgICAibWVtb3J5X2RlcHRoIjogMiwKICAgICAgICAibnVtX3BlcnNpc3RlbnRfdG9rZW5zIjogMCwKICAgICAgICAidGhyZXNob2xkIjogMC4wCiAgICB9CiAgICAKICAgIGlmIHZhcmlhbnRfbmFtZSA9PSAnVGl0YW5NQUMnOgogICAgICAgIGNvbmZpZ1siY2h1bmtfc2l6ZSJdID0gMzIKICAgIGVsc2U6CiAgICAgICAgY29uZmlnWyJ3aW5kb3dfc2l6ZSJdID0gMzIKICAgICAgICAKICAgIHRyeToKICAgICAgICBtb2RlbF9iYXNlID0gTW9kZWxDbGFzcygqKmNvbmZpZykKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIiAgRXJyb3IgaW5zdGFudGlhdGluZyB7dmFyaWFudF9uYW1lfToge2V9IikKICAgICAgICByZXR1cm4KCiAgICAjIFtDUklUSUNBTCBTVEVQXSBQcmUtdHJhaW4gdGhlICdDb250ZXh0IEVuZ2luZScgKEF0dGVudGlvbikKICAgICMgT3RoZXJ3aXNlIEh5YnJpZCBSQUcgZmFpbHMgYmVjYXVzZSBSYW5kb20gV2VpZ2h0cyBkb24ndCBDb3B5LgogICAgcHJldHJhaW5faWRlbnRpdHkobW9kZWxfYmFzZSwgYXJncy5kaW0pCgogICAgIyBbT3B0aW1pemF0aW9uIDIuMF0gSGlnaC1GaWRlbGl0eSBUdW5pbmcKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIG1vZGVsX2Jhc2UubHRtLnRoZXRhLmZpbGxfKDAuMDAxKSAKICAgICAgICBtb2RlbF9iYXNlLmx0bS5hbHBoYS5maWxsXygwLjAwMDEpCiAgICAKICAgIG1vZGVsX3JhZyA9IFRpdGFuUkFHKG1vZGVsX2Jhc2UpCiAgICAKICAgICMgMi4gQmFzZWxpbmUgRXZhbHVhdGlvbgogICAgbmVlZGxlX2RvYyA9IGRvY3NfYmF0Y2hba2V5X2lkeF0KICAgIHRhcmdldCA9IG5lZWRsZV9kb2MKICAgIHByZWRfaW5pdCA9IG1vZGVsX3JhZy50aXRhbi5sdG0uZm9yd2FyZF9ub191cGRhdGUodGFyZ2V0LnJlc2hhcGUoLTEsIGFyZ3MuZGltKSkKICAgIGRpZmZfaW5pdCA9IChwcmVkX2luaXQgLSB0YXJnZXQucmVzaGFwZSgtMSwgYXJncy5kaW0pKQogICAgbG9zc19pbml0ID0gMC41KihkaWZmX2luaXQqKjIpLnN1bSgpLml0ZW0oKQogICAgCiAgICAjIDMuIERpZ2VzdGlvbgogICAgIyBwcmludChmIiAgRGlnZXN0aW5nICgyMDAgZXBvY2hzIHBlciBkb2MpLi4uIikKICAgIGZvciBpLCBkb2MgaW4gZW51bWVyYXRlKGRvY3NfYmF0Y2gpOgogICAgICAgIGZvciBfIGluIHJhbmdlKDIwMCk6IAogICAgICAgICAgICBtb2RlbF9yYWcuZGlnZXN0X2tub3dsZWRnZShkb2MpCiAgICAgICAgICAgIAogICAgIyA0LiBGaW5hbCBFdmFsdWF0aW9uCiAgICBwcmVkX2ZpbmFsID0gbW9kZWxfcmFnLnRpdGFuLmx0bS5mb3J3YXJkX25vX3VwZGF0ZSh0YXJnZXQucmVzaGFwZSgtMSwgYXJncy5kaW0pKQogICAgZGlmZl9maW5hbCA9IChwcmVkX2ZpbmFsIC0gdGFyZ2V0LnJlc2hhcGUoLTEsIGFyZ3MuZGltKSkKICAgIGxvc3NfYWZ0ZXIgPSAwLjUqKGRpZmZfZmluYWwqKjIpLnN1bSgpLml0ZW0oKQogICAgCiAgICAjIDUuIE1ldHJpY3MKICAgIHRvdGFsX2VsZW1lbnRzID0gYXJncy5kb2NfbGVuICogYXJncy5kaW0KICAgIHNpZ25hbF9lbmVyZ3kgPSAoKHRhcmdldC5yZXNoYXBlKC0xLCBhcmdzLmRpbSkpKioyKS5zdW0oKS5pdGVtKCkKICAgIGVycm9yX2VuZXJneSA9IDIgKiBsb3NzX2FmdGVyCiAgICAKICAgIGZpZGVsaXR5ID0gMS4wIC0gKGVycm9yX2VuZXJneSAvIChzaWduYWxfZW5lcmd5ICsgMWUtOSkpCiAgICBmaWRlbGl0eSA9IG1heCgwLjAsIGZpZGVsaXR5KSAqIDEwMC4wCiAgICAKICAgIHByaW50KGYiICBJbml0aWFsIExvc3M6IHtsb3NzX2luaXQ6LjRmfSIpCiAgICBwcmludChmIiAgRmluYWwgTG9zczogICB7bG9zc19hZnRlcjouNGZ9IikKICAgIHByaW50KGYiICBGaWRlbGl0eTogICAgIHtmaWRlbGl0eTouMmZ9JSIpCiAgICAKICAgIGlmIGxvc3NfYWZ0ZXIgPCBsb3NzX2luaXQ6CiAgICAgICAgcHJpbnQoZiIgIFJlc3VsdDogUEFTUyAoSW1wcm92ZW1lbnQgeygobG9zc19pbml0LWxvc3NfYWZ0ZXIpL2xvc3NfaW5pdCkqMTAwOi4xZn0lKSIpCiAgICBlbHNlOgogICAgICAgIHByaW50KGYiICBSZXN1bHQ6IEZBSUwiKQogICAgICAgIAogICAgIyAtLS0gTWV0aG9kIDQ6IEh5YnJpZCBSQUcgKFRoZSBCZXN0IG9mIEJvdGggV29ybGRzKSAtLS0KICAgIHByaW50KGYiXG5be3ZhcmlhbnRfbmFtZX1dIFRlc3RpbmcgSHlicmlkIFJBRyAoV2VpZ2h0ICsgQ29udGV4dCkuLi4iKQogICAgIyBTY2VuYXJpbzogTmVlZGxlIGlzIGluIENvbnRleHQsIE5vaXNlIGlzIGluIFdlaWdodHMuCiAgICAKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICMgSW5wdXQ6IE5lZWRsZSBEb2MgKEJhdGNoPTEsIExlbj1kb2NfbGVuLCBEaW09ZGltKQogICAgICAgICMgQnkgcGFzc2luZyBpdCBhcyBpbnB1dCAneCcgdG8gdGhlIG1vZGVsIGZvcndhcmQgcGFzcywgCiAgICAgICAgIyB0aGUgQXR0ZW50aW9uIG1lY2hhbmlzbSBzZWVzIGl0IGRpcmVjdGx5LgogICAgICAgIGlucCA9IG5lZWRsZV9kb2MgIyBBbHJlYWR5IGhhcyBiYXRjaCBkaW0gWzEsIDMyLCA2NF0KICAgICAgICBvdXRwdXQgPSBtb2RlbF9yYWcoaW5wKQogICAgICAgIAogICAgZGlmZl9oeWJyaWQgPSAob3V0cHV0IC0gaW5wKQogICAgbG9zc19oeWJyaWQgPSAwLjUqKGRpZmZfaHlicmlkKioyKS5zdW0oKS5pdGVtKCkKICAgIAogICAgZXJyb3JfZW5lcmd5X2h5YiA9IDIgKiBsb3NzX2h5YnJpZAogICAgZmlkZWxpdHlfaHliID0gMS4wIC0gKGVycm9yX2VuZXJneV9oeWIgLyAoc2lnbmFsX2VuZXJneSArIDFlLTkpKQogICAgZmlkZWxpdHlfaHliID0gbWF4KDAuMCwgZmlkZWxpdHlfaHliKSAqIDEwMC4wCiAgICAKICAgIHByaW50KGYiICBIeWJyaWQgRmlkZWxpdHk6IHtmaWRlbGl0eV9oeWI6LjJmfSUgKFNob3VsZCBiZSB+MTAwJSkiKQogICAgaWYgZmlkZWxpdHlfaHliID4gOTkuMDoKICAgICAgICBwcmludChmIiAgUmVzdWx0OiBQQVNTIChQZXJmZWN0IFJlY2FsbCB2aWEgQXR0ZW50aW9uKSIpCgpkZWYgY29tcGFyZV9tZXRob2RzKGFyZ3MpOgogICAgcHJpbnQoIi0tLSBDb25kdWN0aW5nIFJBRyBDb21wYXJpc29uIEV4cGVyaW1lbnQgKEFsbCBWYXJpYW50cykgLS0tIikKICAgIHByaW50KGYiRG9jczoge2FyZ3MubnVtX2RvY3N9LCBEb2MgTGVuOiB7YXJncy5kb2NfbGVufSwgRGltOiB7YXJncy5kaW19IikKICAgIAogICAgIyAxLiBTZXR1cCBEYXRhCiAgICBkb2NzLCBfLCBrZXlfaWR4ID0gZ2VuZXJhdGVfaGF5c3RhY2soYXJncy5udW1fZG9jcywgYXJncy5kb2NfbGVuLCBhcmdzLmRpbSkKICAgIGRvY3NfYmF0Y2ggPSBbZC51bnNxdWVlemUoMCkgZm9yIGQgaW4gZG9jc10KICAgIAogICAgIyAyLiBUcmFkaXRpb25hbCBSQUcgQmFzZWxpbmUgKE9ubHkgZm9yIE1BQywgYXMgY29udGV4dCBjb25jYXQgaXMgdW5pcXVlIHRvIGl0IGNvbmNlcHR1YWxseSkKICAgIHByaW50KCJcbltCYXNlbGluZTogVHJhZGl0aW9uYWwgUkFHIChDb25jYXRlbmF0aW9uKV0iKQogICAgIyBPbmx5IE1BQyBpcyByZWFsbHkgc3VpdGFibGUgZm9yICdDb25jYXQgYWxsJyBuYWl2ZSBSQUcsIG90aGVycyB1c2Ugc2xpZGluZyB3aW5kb3cvZ2F0aW5nLgogICAgIyBCdXQgZm9yIE9PTSB0ZXN0aW5nLCBhbnkgbW9kZWwgcmVjZWl2aW5nIGh1Z2UgaW5wdXQgd29ya3MuCiAgICB0cnk6CiAgICAgICAgZnVsbF9jb250ZXh0ID0gdG9yY2guY2F0KGRvY3NfYmF0Y2gsIGRpbT0xKQogICAgICAgIGlmIGZ1bGxfY29udGV4dC5zaXplKDEpID4gMjA0ODoKICAgICAgICAgICAgcHJpbnQoZiIgIFJlc3VsdDogRkFJTCAoQ29udGV4dCBMZW5ndGgge2Z1bGxfY29udGV4dC5zaXplKDEpfSA+IDIwNDgpIikKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcmludChmIiAgUmVzdWx0OiBQQVNTIChQcm9jZXNzZWQge2Z1bGxfY29udGV4dC5zaXplKDEpfSB0b2tlbnMpIikKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgcHJpbnQoZiIgIFJlc3VsdDogRkFJTCAoe2V9KSIpCgogICAgIyAzLiBUaXRhblJBRyBWYXJpYW50cwogICAgdmFyaWFudHMgPSBbCiAgICAgICAgKCdUaXRhbk1BQycsIFRpdGFuTUFDKSwKICAgICAgICAoJ1RpdGFuTUFHJywgVGl0YW5NQUcpLAogICAgICAgICgnVGl0YW5NQUwnLCBUaXRhbk1BTCkKICAgIF0KICAgIAogICAgZm9yIG5hbWUsIGNscyBpbiB2YXJpYW50czoKICAgICAgICBydW5fc2luZ2xlX3ZhcmlhbnQobmFtZSwgY2xzLCBhcmdzLCBkb2NzX2JhdGNoLCBrZXlfaWR4KQoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbnVtX2RvY3MiLCB0eXBlPWludCwgZGVmYXVsdD01MCwgaGVscD0iTnVtYmVyIG9mIHJldHJpZXZlZCBkb2N1bWVudHMiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kb2NfbGVuIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NjQsIGhlbHA9Ikxlbmd0aCBvZiBlYWNoIGRvY3VtZW50IikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGltIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NjQpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgCiAgICBjb21wYXJlX21ldGhvZHMoYXJncykK',
    'data/sample_essays.py': 'IiIiClNhbXBsZSBFc3NheXMgZm9yIFRpdGFuUkFHIEV4cGVyaW1lbnRzCgpUaGVzZSBlc3NheXMgY29udGFpbiBmYWN0dWFsIGluZm9ybWF0aW9uIHRoYXQgY2FuIGJlIHVzZWQgdG8gdGVzdAp0aGUgbW9kZWwncyBhYmlsaXR5IHRvIG1lbW9yaXplIGFuZCByZWNhbGwgc3BlY2lmaWMgZGV0YWlscy4KIiIiCgpFU1NBWV9DTElNQVRFID0gIiIiCiMgVGhlIFNjaWVuY2Ugb2YgQ2xpbWF0ZSBDaGFuZ2UKCiMjIEludHJvZHVjdGlvbgpDbGltYXRlIGNoYW5nZSByZWZlcnMgdG8gbG9uZy10ZXJtIHNoaWZ0cyBpbiBnbG9iYWwgdGVtcGVyYXR1cmVzIGFuZCB3ZWF0aGVyIHBhdHRlcm5zLiAKV2hpbGUgbmF0dXJhbCBmYWN0b3JzIGxpa2Ugdm9sY2FuaWMgZXJ1cHRpb25zIGFuZCBzb2xhciBjeWNsZXMgY2FuIGluZmx1ZW5jZSBjbGltYXRlLCAKaHVtYW4gYWN0aXZpdGllcyBoYXZlIGJlZW4gdGhlIHByaW1hcnkgZHJpdmVyIHNpbmNlIHRoZSAxODAwcy4KCiMjIEtleSBGYWN0cwotIFRoZSBnbG9iYWwgYXZlcmFnZSB0ZW1wZXJhdHVyZSBoYXMgcmlzZW4gYnkgYXBwcm94aW1hdGVseSAxLjHCsEMgc2luY2UgcHJlLWluZHVzdHJpYWwgdGltZXMuCi0gVGhlIGNvbmNlbnRyYXRpb24gb2YgQ08yIGluIHRoZSBhdG1vc3BoZXJlIHJlYWNoZWQgNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMuCi0gU2VhIGxldmVscyBoYXZlIHJpc2VuIGJ5IGFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykgc2luY2UgMTkwMC4KLSBUaGUgQXJjdGljIGlzIHdhcm1pbmcgbmVhcmx5IGZvdXIgdGltZXMgZmFzdGVyIHRoYW4gdGhlIGdsb2JhbCBhdmVyYWdlLgoKIyMgQ2F1c2VzClRoZSBidXJuaW5nIG9mIGZvc3NpbCBmdWVsc+KAlGNvYWwsIG9pbCwgYW5kIG5hdHVyYWwgZ2Fz4oCUcmVsZWFzZXMgY2FyYm9uIGRpb3hpZGUgKENPMikgCmFuZCBvdGhlciBncmVlbmhvdXNlIGdhc2VzIGludG8gdGhlIGF0bW9zcGhlcmUuIFRoZXNlIGdhc2VzIHRyYXAgaGVhdCBmcm9tIHRoZSBzdW4sIApjcmVhdGluZyBhICJncmVlbmhvdXNlIGVmZmVjdCIgdGhhdCB3YXJtcyB0aGUgcGxhbmV0LgoKIyMgSW1wYWN0cwoxLiAqKkV4dHJlbWUgV2VhdGhlcioqOiBNb3JlIGZyZXF1ZW50IGhlYXR3YXZlcywgZHJvdWdodHMsIGFuZCBpbnRlbnNlIHN0b3Jtcy4KMi4gKipFY29zeXN0ZW0gRGlzcnVwdGlvbioqOiBDb3JhbCBibGVhY2hpbmcsIHNwZWNpZXMgbWlncmF0aW9uLCBhbmQgYmlvZGl2ZXJzaXR5IGxvc3MuCjMuICoqSHVtYW4gSGVhbHRoKio6IEluY3JlYXNlZCBoZWF0LXJlbGF0ZWQgaWxsbmVzc2VzIGFuZCBzcHJlYWQgb2YgdmVjdG9yLWJvcm5lIGRpc2Vhc2VzLgo0LiAqKkFncmljdWx0dXJlKio6IFNoaWZ0cyBpbiBncm93aW5nIHNlYXNvbnMgYW5kIHJlZHVjZWQgY3JvcCB5aWVsZHMgaW4gc29tZSByZWdpb25zLgoKIyMgU29sdXRpb25zCi0gVHJhbnNpdGlvbiB0byByZW5ld2FibGUgZW5lcmd5IHNvdXJjZXMgKHNvbGFyLCB3aW5kLCBoeWRybykuCi0gSW1wcm92ZSBlbmVyZ3kgZWZmaWNpZW5jeSBpbiBidWlsZGluZ3MgYW5kIHRyYW5zcG9ydGF0aW9uLgotIFByb3RlY3QgYW5kIHJlc3RvcmUgZm9yZXN0cywgd2hpY2ggYWJzb3JiIENPMi4KLSBJbXBsZW1lbnQgY2FyYm9uIHByaWNpbmcgdG8gaW5jZW50aXZpemUgZW1pc3Npb24gcmVkdWN0aW9ucy4KCiMjIENvbmNsdXNpb24KQWRkcmVzc2luZyBjbGltYXRlIGNoYW5nZSByZXF1aXJlcyB1cmdlbnQsIGNvb3JkaW5hdGVkIGdsb2JhbCBhY3Rpb24uIFRoZSBQYXJpcyBBZ3JlZW1lbnQgCm9mIDIwMTUgc2V0IHRoZSBnb2FsIG9mIGxpbWl0aW5nIHdhcm1pbmcgdG8gMS41wrBDIGFib3ZlIHByZS1pbmR1c3RyaWFsIGxldmVscy4KIiIiCgpFU1NBWV9BSSA9ICIiIgojIFRoZSBFdm9sdXRpb24gb2YgQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UKCiMjIEhpc3RvcmljYWwgQmFja2dyb3VuZApUaGUgdGVybSAiQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UiIHdhcyBjb2luZWQgYnkgSm9obiBNY0NhcnRoeSBpbiAxOTU2IGF0IHRoZSBEYXJ0bW91dGggCkNvbmZlcmVuY2UuIEVhcmx5IEFJIHJlc2VhcmNoIGZvY3VzZWQgb24gc3ltYm9saWMgcmVhc29uaW5nIGFuZCBwcm9ibGVtLXNvbHZpbmcuCgojIyBUaW1lbGluZSBvZiBNYWpvciBNaWxlc3RvbmVzCi0gKioxOTUwKio6IEFsYW4gVHVyaW5nIHByb3Bvc2VzIHRoZSAiVHVyaW5nIFRlc3QiIGZvciBtYWNoaW5lIGludGVsbGlnZW5jZS4KLSAqKjE5NTYqKjogRGFydG1vdXRoIENvbmZlcmVuY2UgbWFya3MgdGhlIGJpcnRoIG9mIEFJIGFzIGEgZmllbGQuCi0gKioxOTk3Kio6IElCTSdzIERlZXAgQmx1ZSBkZWZlYXRzIHdvcmxkIGNoZXNzIGNoYW1waW9uIEdhcnJ5IEthc3Bhcm92LgotICoqMjAxMSoqOiBJQk0gV2F0c29uIHdpbnMgSmVvcGFyZHkhIGFnYWluc3QgaHVtYW4gY2hhbXBpb25zLgotICoqMjAxNioqOiBHb29nbGUncyBBbHBoYUdvIGRlZmVhdHMgTGVlIFNlZG9sIGluIHRoZSBnYW1lIG9mIEdvLgotICoqMjAyMioqOiBDaGF0R1BUIGxhdW5jaGVzLCBkZW1vbnN0cmF0aW5nIGFkdmFuY2VkIGNvbnZlcnNhdGlvbmFsIEFJLgoKIyMgVHlwZXMgb2YgQUkKMS4gKipOYXJyb3cgQUkgKEFOSSkqKjogRGVzaWduZWQgZm9yIHNwZWNpZmljIHRhc2tzIChlLmcuLCBpbWFnZSByZWNvZ25pdGlvbiwgbGFuZ3VhZ2UgdHJhbnNsYXRpb24pLgoyLiAqKkdlbmVyYWwgQUkgKEFHSSkqKjogSHlwb3RoZXRpY2FsIEFJIHdpdGggaHVtYW4tbGV2ZWwgY29nbml0aXZlIGFiaWxpdGllcyBhY3Jvc3MgYWxsIGRvbWFpbnMuCjMuICoqU3VwZXJpbnRlbGxpZ2VudCBBSSAoQVNJKSoqOiBUaGVvcmV0aWNhbCBBSSBzdXJwYXNzaW5nIGh1bWFuIGludGVsbGlnZW5jZSBlbnRpcmVseS4KCiMjIEN1cnJlbnQgQXBwbGljYXRpb25zCi0gKipIZWFsdGhjYXJlKio6IE1lZGljYWwgaW1hZ2luZyBhbmFseXNpcywgZHJ1ZyBkaXNjb3ZlcnksIHBlcnNvbmFsaXplZCB0cmVhdG1lbnQgcGxhbnMuCi0gKipGaW5hbmNlKio6IEZyYXVkIGRldGVjdGlvbiwgYWxnb3JpdGhtaWMgdHJhZGluZywgY3JlZGl0IHNjb3JpbmcuCi0gKipUcmFuc3BvcnRhdGlvbioqOiBBdXRvbm9tb3VzIHZlaGljbGVzLCB0cmFmZmljIG9wdGltaXphdGlvbiwgcm91dGUgcGxhbm5pbmcuCi0gKipFZHVjYXRpb24qKjogUGVyc29uYWxpemVkIGxlYXJuaW5nLCBhdXRvbWF0ZWQgZ3JhZGluZywgdHV0b3Jpbmcgc3lzdGVtcy4KCiMjIEtleSBUZWNobm9sb2dpZXMKVGhlIG1vZGVybiBBSSByZXZvbHV0aW9uIGlzIGJ1aWx0IG9uOgotICoqRGVlcCBMZWFybmluZyoqOiBOZXVyYWwgbmV0d29ya3Mgd2l0aCBtdWx0aXBsZSBsYXllcnMgZm9yIGZlYXR1cmUgZXh0cmFjdGlvbi4KLSAqKlRyYW5zZm9ybWVycyoqOiBBcmNoaXRlY3R1cmUgYmVoaW5kIGxhcmdlIGxhbmd1YWdlIG1vZGVscyBsaWtlIEdQVC00IGFuZCBDbGF1ZGUuCi0gKipSZWluZm9yY2VtZW50IExlYXJuaW5nKio6IFRyYWluaW5nIGFnZW50cyB0aHJvdWdoIHJld2FyZC1iYXNlZCBmZWVkYmFjay4KCiMjIEV0aGljYWwgQ29uc2lkZXJhdGlvbnMKLSBCaWFzIGluIHRyYWluaW5nIGRhdGEgbGVhZGluZyB0byB1bmZhaXIgb3V0Y29tZXMuCi0gSm9iIGRpc3BsYWNlbWVudCBhbmQgZWNvbm9taWMgaW5lcXVhbGl0eS4KLSBQcml2YWN5IGNvbmNlcm5zIGZyb20gZGF0YSBjb2xsZWN0aW9uLgotIFBvdGVudGlhbCBtaXN1c2UgZm9yIG1pc2luZm9ybWF0aW9uIGFuZCBzdXJ2ZWlsbGFuY2UuCgojIyBUaGUgRnV0dXJlCkV4cGVydHMgcHJlZGljdCBBSSB3aWxsIGNvbnRpbnVlIHRvIHRyYW5zZm9ybSBpbmR1c3RyaWVzLCB3aXRoIGVzdGltYXRlZCBlY29ub21pYyBpbXBhY3QgCm9mICQxNS43IHRyaWxsaW9uIGJ5IDIwMzAgYWNjb3JkaW5nIHRvIFB3Qy4gVGhlIGRldmVsb3BtZW50IG9mIEFHSSByZW1haW5zIGEgbG9uZy10ZXJtIGdvYWwuCiIiIgoKRVNTQVlfU1BBQ0UgPSAiIiIKIyBFeHBsb3JpbmcgdGhlIFNvbGFyIFN5c3RlbQoKIyMgT3VyIENvc21pYyBOZWlnaGJvcmhvb2QKVGhlIFNvbGFyIFN5c3RlbSBjb25zaXN0cyBvZiB0aGUgU3VuLCBlaWdodCBwbGFuZXRzLCBkd2FyZiBwbGFuZXRzLCBtb29ucywgYXN0ZXJvaWRzLCAKYW5kIGNvbWV0cy4gSXQgZm9ybWVkIGFwcHJveGltYXRlbHkgNC42IGJpbGxpb24geWVhcnMgYWdvIGZyb20gYSBnaWFudCBtb2xlY3VsYXIgY2xvdWQuCgojIyBUaGUgU3VuCi0gKipUeXBlKio6IEctdHlwZSBtYWluLXNlcXVlbmNlIHN0YXIgKHllbGxvdyBkd2FyZikKLSAqKkRpYW1ldGVyKio6IDEsMzkyLDcwMCBrbSAoMTA5IHRpbWVzIEVhcnRoJ3MgZGlhbWV0ZXIpCi0gKipTdXJmYWNlIFRlbXBlcmF0dXJlKio6IDUsNTAwwrBDICg5LDkwMMKwRikKLSAqKkNvcmUgVGVtcGVyYXR1cmUqKjogMTUgbWlsbGlvbsKwQwotICoqRGlzdGFuY2UgZnJvbSBFYXJ0aCoqOiAxNDkuNiBtaWxsaW9uIGttICgxIEFzdHJvbm9taWNhbCBVbml0KQoKIyMgVGhlIFBsYW5ldHMKIyMjIElubmVyIFBsYW5ldHMgKFJvY2t5L1RlcnJlc3RyaWFsKQoxLiAqKk1lcmN1cnkqKjogU21hbGxlc3QgcGxhbmV0LCA4OC1kYXkgb3JiaXQsIG5vIG1vb25zLgoyLiAqKlZlbnVzKio6IEhvdHRlc3QgcGxhbmV0ICg0NzXCsEMgc3VyZmFjZSksIHJvdGF0ZXMgYmFja3dhcmRzLgozLiAqKkVhcnRoKio6IE9ubHkga25vd24gcGxhbmV0IHdpdGggbGlxdWlkIHdhdGVyIG9uIHN1cmZhY2UgYW5kIGxpZmUuCjQuICoqTWFycyoqOiAiUmVkIFBsYW5ldCIsIGhhcyB0aGUgbGFyZ2VzdCB2b2xjYW5vIChPbHltcHVzIE1vbnMsIDIxLjkga20gaGlnaCkuCgojIyMgT3V0ZXIgUGxhbmV0cyAoR2FzL0ljZSBHaWFudHMpCjUuICoqSnVwaXRlcioqOiBMYXJnZXN0IHBsYW5ldCAoMTF4IEVhcnRoJ3MgZGlhbWV0ZXIpLCBHcmVhdCBSZWQgU3BvdCBzdG9ybS4KNi4gKipTYXR1cm4qKjogRmFtb3VzIGZvciBpdHMgcmluZyBzeXN0ZW0sIDgzIGtub3duIG1vb25zIGluY2x1ZGluZyBUaXRhbi4KNy4gKipVcmFudXMqKjogUm90YXRlcyBvbiBpdHMgc2lkZSAoOTjCsCBheGlhbCB0aWx0KSwgMjcgbW9vbnMuCjguICoqTmVwdHVuZSoqOiBTdHJvbmdlc3Qgd2luZHMgaW4gdGhlIFNvbGFyIFN5c3RlbSAoMiwxMDAga20vaCkuCgojIyBOb3RhYmxlIE1pc3Npb25zCi0gKipWb3lhZ2VyIDEgJiAyICgxOTc3KSoqOiBGaXJzdCBzcGFjZWNyYWZ0IHRvIHJlYWNoIGludGVyc3RlbGxhciBzcGFjZS4KLSAqKkh1YmJsZSBTcGFjZSBUZWxlc2NvcGUgKDE5OTApKio6IFJldm9sdXRpb25pemVkIG91ciB2aWV3IG9mIHRoZSB1bml2ZXJzZS4KLSAqKk1hcnMgUm92ZXJzKio6IFNwaXJpdCwgT3Bwb3J0dW5pdHksIEN1cmlvc2l0eSwgUGVyc2V2ZXJhbmNlIGV4cGxvcmVkIE1hcnMuCi0gKipKYW1lcyBXZWJiIFNwYWNlIFRlbGVzY29wZSAoMjAyMSkqKjogTW9zdCBwb3dlcmZ1bCBzcGFjZSB0ZWxlc2NvcGUgZXZlciBidWlsdC4KCiMjIER3YXJmIFBsYW5ldHMKUGx1dG8gd2FzIHJlY2xhc3NpZmllZCBhcyBhIGR3YXJmIHBsYW5ldCBpbiAyMDA2IGJ5IHRoZSBJbnRlcm5hdGlvbmFsIEFzdHJvbm9taWNhbCBVbmlvbi4gCk90aGVyIGR3YXJmIHBsYW5ldHMgaW5jbHVkZSBFcmlzLCBIYXVtZWEsIE1ha2VtYWtlLCBhbmQgQ2VyZXMuCgojIyBGdXR1cmUgb2YgU3BhY2UgRXhwbG9yYXRpb24KLSAqKjIwMjQtMjAyNSoqOiBBcnRlbWlzIHByb2dyYW0gYWltcyB0byByZXR1cm4gaHVtYW5zIHRvIHRoZSBNb29uLgotICoqMjAzMHMqKjogTkFTQSBhbmQgU3BhY2VYIHBsYW5uaW5nIGNyZXdlZCBtaXNzaW9ucyB0byBNYXJzLgotICoqTG9uZy10ZXJtKio6IFBvdGVudGlhbCBjb2xvbml6YXRpb24gb2YgTWFycyBhbmQgYXN0ZXJvaWQgbWluaW5nLgoiIiIKCiMgUXVlc3Rpb25zIGZvciB0ZXN0aW5nIC0gTWl4ZWQgZmFjdHVhbCBhbmQgcmVhc29uaW5nIHF1ZXN0aW9ucwojIEZhY3R1YWwgcXVlc3Rpb25zICh3aGF0L3doZW4vd2hlcmUpIHNob3VsZCB0cmlnZ2VyIEhZQlJJRCBtb2RlCiMgUmVhc29uaW5nIHF1ZXN0aW9ucyAod2h5L2V4cGxhaW4vZGVzY3JpYmUpIHNob3VsZCB0cmlnZ2VyIFRJVEFOU19PTkxZIG1vZGUKVEVTVF9RVUVTVElPTlMgPSB7CiAgICAiY2xpbWF0ZSI6IFsKICAgICAgICAjIEZhY3R1YWwgcXVlc3Rpb25zIChleHBlY3QgSFlCUklEKQogICAgICAgICgiV2hhdCBpcyB0aGUgY3VycmVudCBDTzIgY29uY2VudHJhdGlvbiBpbiB0aGUgYXRtb3NwaGVyZT8iLCAiNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMiKSwKICAgICAgICAoIkhvdyBtdWNoIGhhcyB0aGUgZ2xvYmFsIHRlbXBlcmF0dXJlIHJpc2VuIHNpbmNlIHByZS1pbmR1c3RyaWFsIHRpbWVzPyIsICJhcHByb3hpbWF0ZWx5IDEuMcKwQyIpLAogICAgICAgICgiV2hhdCB5ZWFyIHdhcyB0aGUgUGFyaXMgQWdyZWVtZW50IHNpZ25lZD8iLCAiMjAxNSIpLAogICAgICAgICgiSG93IG11Y2ggaGF2ZSBzZWEgbGV2ZWxzIHJpc2VuIHNpbmNlIDE5MDA/IiwgImFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykiKSwKICAgICAgICAKICAgICAgICAjIFJlYXNvbmluZyBxdWVzdGlvbnMgKGV4cGVjdCBUSVRBTlNfT05MWSkKICAgICAgICAoIldoeSBpcyBjbGltYXRlIGNoYW5nZSBwcmltYXJpbHkgY2F1c2VkIGJ5IGh1bWFuIGFjdGl2aXRpZXM/IiwgImJ1cm5pbmcgb2YgZm9zc2lsIGZ1ZWxzIHJlbGVhc2VzIGdyZWVuaG91c2UgZ2FzZXMgdGhhdCB0cmFwIGhlYXQiKSwKICAgICAgICAoIkV4cGxhaW4gaG93IHRoZSBncmVlbmhvdXNlIGVmZmVjdCB3b3JrcyIsICJncmVlbmhvdXNlIGdhc2VzIHRyYXAgaGVhdCBmcm9tIHRoZSBzdW4sIGNyZWF0aW5nIHdhcm1pbmcgZWZmZWN0IiksCiAgICAgICAgKCJEZXNjcmliZSB0aGUgbWFpbiBpbXBhY3RzIG9mIGNsaW1hdGUgY2hhbmdlIG9uIGVjb3N5c3RlbXMiLCAiY29yYWwgYmxlYWNoaW5nLCBzcGVjaWVzIG1pZ3JhdGlvbiwgYmlvZGl2ZXJzaXR5IGxvc3MsIGFuZCBoYWJpdGF0IGRpc3J1cHRpb24iKSwKICAgIF0sCiAgICAiYWkiOiBbCiAgICAgICAgIyBGYWN0dWFsIHF1ZXN0aW9ucyAoZXhwZWN0IEhZQlJJRCkKICAgICAgICAoIldobyBjb2luZWQgdGhlIHRlcm0gJ0FydGlmaWNpYWwgSW50ZWxsaWdlbmNlJz8iLCAiSm9obiBNY0NhcnRoeSBpbiAxOTU2IiksCiAgICAgICAgKCJXaGVuIGRpZCBEZWVwIEJsdWUgZGVmZWF0IEdhcnJ5IEthc3Bhcm92PyIsICIxOTk3IiksCiAgICAgICAgKCJXaGF0IGlzIHRoZSBlc3RpbWF0ZWQgZWNvbm9taWMgaW1wYWN0IG9mIEFJIGJ5IDIwMzA/IiwgIiQxNS43IHRyaWxsaW9uIGFjY29yZGluZyB0byBQd0MiKSwKICAgICAgICAoIldoYXQgYXJjaGl0ZWN0dXJlIHBvd2VycyBHUFQtNCBhbmQgQ2xhdWRlPyIsICJUcmFuc2Zvcm1lcnMiKSwKICAgICAgICAKICAgICAgICAjIFJlYXNvbmluZyBxdWVzdGlvbnMgKGV4cGVjdCBUSVRBTlNfT05MWSkKICAgICAgICAoIldoeSBpcyBiaWFzIGluIEFJIGEgbWFqb3IgZXRoaWNhbCBjb25jZXJuPyIsICJiaWFzIGluIHRyYWluaW5nIGRhdGEgY2FuIGxlYWQgdG8gdW5mYWlyIG91dGNvbWVzIGFuZCBkaXNjcmltaW5hdGlvbiIpLAogICAgICAgICgiRXhwbGFpbiB0aGUgZGlmZmVyZW5jZSBiZXR3ZWVuIE5hcnJvdyBBSSBhbmQgR2VuZXJhbCBBSSIsICJOYXJyb3cgQUkgaXMgZGVzaWduZWQgZm9yIHNwZWNpZmljIHRhc2tzIHdoaWxlIEdlbmVyYWwgQUkgaGFzIGh1bWFuLWxldmVsIGNvZ25pdGl2ZSBhYmlsaXRpZXMgYWNyb3NzIGFsbCBkb21haW5zIiksCiAgICAgICAgKCJEZXNjcmliZSBob3cgcmVpbmZvcmNlbWVudCBsZWFybmluZyB3b3JrcyIsICJ0cmFpbmluZyBhZ2VudHMgdGhyb3VnaCByZXdhcmQtYmFzZWQgZmVlZGJhY2sgdG8gbGVhcm4gb3B0aW1hbCBiZWhhdmlvcnMiKSwKICAgIF0sCiAgICAic3BhY2UiOiBbCiAgICAgICAgIyBGYWN0dWFsIHF1ZXN0aW9ucyAoZXhwZWN0IEhZQlJJRCkKICAgICAgICAoIkhvdyBvbGQgaXMgdGhlIFNvbGFyIFN5c3RlbT8iLCAiYXBwcm94aW1hdGVseSA0LjYgYmlsbGlvbiB5ZWFycyIpLAogICAgICAgICgiV2hhdCBpcyB0aGUgdGFsbGVzdCB2b2xjYW5vIGluIHRoZSBTb2xhciBTeXN0ZW0/IiwgIk9seW1wdXMgTW9ucywgMjEuOSBrbSBoaWdoIiksCiAgICAgICAgKCJXaGVuIHdhcyB0aGUgSmFtZXMgV2ViYiBTcGFjZSBUZWxlc2NvcGUgbGF1bmNoZWQ/IiwgIjIwMjEiKSwKICAgICAgICAoIldoYXQgeWVhciB3YXMgUGx1dG8gcmVjbGFzc2lmaWVkIGFzIGEgZHdhcmYgcGxhbmV0PyIsICIyMDA2IiksCiAgICAgICAgCiAgICAgICAgIyBSZWFzb25pbmcgcXVlc3Rpb25zIChleHBlY3QgVElUQU5TX09OTFkpCiAgICAgICAgKCJXaHkgaXMgVmVudXMgdGhlIGhvdHRlc3QgcGxhbmV0IGRlc3BpdGUgTWVyY3VyeSBiZWluZyBjbG9zZXIgdG8gdGhlIFN1bj8iLCAiVmVudXMgaGFzIGEgdGhpY2sgYXRtb3NwaGVyZSB0aGF0IHRyYXBzIGhlYXQgdGhyb3VnaCBncmVlbmhvdXNlIGVmZmVjdCIpLAogICAgICAgICgiRXhwbGFpbiB3aHkgb3V0ZXIgcGxhbmV0cyBhcmUgY2FsbGVkIGdhcyBnaWFudHMiLCAidGhleSBhcmUgbXVjaCBsYXJnZXIgYW5kIGNvbXBvc2VkIHByaW1hcmlseSBvZiBnYXNlcyBsaWtlIGh5ZHJvZ2VuIGFuZCBoZWxpdW0gcmF0aGVyIHRoYW4gc29saWQgcm9jayIpLAogICAgICAgICgiRGVzY3JpYmUgdGhlIHNpZ25pZmljYW5jZSBvZiB0aGUgVm95YWdlciBtaXNzaW9ucyIsICJmaXJzdCBzcGFjZWNyYWZ0IHRvIHJlYWNoIGludGVyc3RlbGxhciBzcGFjZSBhbmQgZXhwbG9yZSB0aGUgb3V0ZXIgc29sYXIgc3lzdGVtIiksCiAgICBdLAp9CgpkZWYgZ2V0X2FsbF9lc3NheXMoKToKICAgICIiIlJldHVybiBhbGwgZXNzYXlzIGFzIGEgZGljdGlvbmFyeS4iIiIKICAgIHJldHVybiB7CiAgICAgICAgImNsaW1hdGUiOiBFU1NBWV9DTElNQVRFLAogICAgICAgICJhaSI6IEVTU0FZX0FJLAogICAgICAgICJzcGFjZSI6IEVTU0FZX1NQQUNFLAogICAgfQoKZGVmIGdldF90ZXN0X3F1ZXN0aW9ucygpOgogICAgIiIiUmV0dXJuIHRlc3QgcXVlc3Rpb25zIHdpdGggZXhwZWN0ZWQgYW5zd2Vycy4iIiIKICAgIHJldHVybiBURVNUX1FVRVNUSU9OUwo=',
    'projects/hybrid_titans/essay_rag_demo.py': '...',
    'projects/hybrid_titans/hybrid_titan_demo.py': '...'
}

for path, content in files.items():
    write_file(path, content)

print('Files in current directory:', os.listdir('.'))
if os.path.exists('experiments'):
    print('Files in experiments:', os.listdir('experiments'))


## 3. Run Baseline (Dense Update)

In [ ]:
# Run Baseline (Dense Update)
# Decreasing vocab_size to 16 to ensure model can learn quickly
import os
if os.path.exists('projects/original_benchmarks/train_toy.py'):
    print('Running Baseline (Dense Update, Vocab=16)...')
    !python projects/original_benchmarks/train_toy.py --model_variant MAC --num_epochs 5 --batch_size 16 --vocab_size 16 --threshold 0.0
else:
    print('ERROR: projects/original_benchmarks/train_toy.py not found!')


## 4. Run Innovation (Sparse Update)

In [ ]:
# Run Innovation (Sparse Update)
print('Running Sparse Update (Threshold = 0.5, Vocab=16)...')
!python projects/original_benchmarks/train_toy.py --model_variant MAC --num_epochs 5 --batch_size 16 --vocab_size 16 --threshold 0.5


## 5. TitanRAG Prototype Demo

In [ ]:
# TitanRAG Prototype Demo
# Demonstrating 'Retrieve-then-Digest' mechanism
import torch
import sys, os
sys.path.append(os.path.abspath('src'))
from main import TitanMAC, TitanRAG

print('Initializing TitanRAG...')
# 1. Initialize Base Model
base_model = TitanMAC(dim=128, chunk_size=32, hidden_dim=128, memory_depth=2, num_persistent_tokens=4, threshold=0.1)
rag_model = TitanRAG(base_model)

# 2. Simulate Data (Embeddings)
print('Simulating Retrieval...')
doc1 = torch.randn(1, 64, 128) # Retrieved Doc A
doc2 = torch.randn(1, 64, 128) # Retrieved Doc B
query = torch.randn(1, 16, 128) # User Query

# 3. Run Pipeline
print('Executing Digestion & Inference...')
output = rag_model.query_with_context(query, [doc1, doc2])

print('Success! Output shape:', output.shape)
print('The model weights have been updated on-the-fly with document info.')


## 6. RAG Comparison Experiment (Small Scale)

In [ ]:
# Experiment: RAG Comparison - Small Scale (Within Context Window)
# Both methods should succeed.
print('--- Running RAG Comparison (Small Scale: 10 Docs) ---')
!python projects/hybrid_titans/rag_compare.py --num_docs 10 --doc_len 32


## 7. RAG Comparison Experiment (Large Scale)

In [ ]:
# Experiment: RAG Comparison - Large Scale (Exceeds Context Window)
# Traditional RAG should FAIL (Context Limit).
# TitanRAG should PASS (Infinite Memory).
print('\n--- Running RAG Comparison (Large Scale: 100 Docs) ---')
!python projects/hybrid_titans/rag_compare.py --num_docs 100 --doc_len 32


## 8. Essay-based TitanRAG Demo
This cell demonstrates TitanRAG on real text: Climate Essay.
The model digests the essay and then answers verification questions.

In [ ]:
# Run Essay RAG Demo (Climate Essay - Random Embedder Baseline)
print('--- Essay RAG Demo: Climate (Random Embedder) ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay climate --epochs 50 --embedder random


## 9. Semantic Embedder (sentence-transformers)
Now running with **semantic embeddings** for meaningful text similarity.
This should show significant improvement in Hybrid RAG performance!

In [ ]:
# Run ALL Essays with Semantic Embedder
print('=' * 60)
print('SEMANTIC EMBEDDER TEST - ALL ESSAYS')
print('=' * 60)

# Climate Essay
print('\n--- Essay 1/3: Climate Change ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay climate --epochs 50 --embedder semantic

# AI Essay
print('\n--- Essay 2/3: Artificial Intelligence ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay ai --epochs 50 --embedder semantic

# Space Essay
print('\n--- Essay 3/3: Solar System ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay space --epochs 50 --embedder semantic


## 10. Hybrid Titan: TitanRAG + Flan-T5 + Decision Maker
This cell demonstrates **Direction A** with intelligent mode selection:
- **Decision Maker** automatically chooses between titans_only and hybrid modes
- TitanRAG performs retrieval using Ensemble Fusion (Keyword + Memory + Embedding)
- Flan-T5 generates natural language answers from context
- **Online Learning**: Model learns from each Q&A pair

**Decision Signals**:
1. Question Type (30%): Factual vs Reasoning
2. Memory Confidence (40%): LTM output strength
3. Retrieval Dispersion (30%): Score distribution

In [ ]:
# Hybrid Titan Demo: TitanRAG Retrieval + Flan-T5 Generation + Decision Maker
# Running ALL THREE ESSAYS with Online Learning

print('=' * 60)
print('HYBRID TITAN DEMO - ALL ESSAYS')
print('=' * 60)

# 1. Climate Essay (Default Decision Maker Settings)
print('\n' + '─' * 60)
print('📖 Essay 1/3: CLIMATE CHANGE (Default Settings)')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay climate --epochs 50 --topk 5

# 2. AI Essay (Custom: More aggressive titans_only)
print('\n' + '─' * 60)
print('📖 Essay 2/3: ARTIFICIAL INTELLIGENCE (High Threshold)')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay ai --epochs 50 --topk 5 --decision_threshold 0.6

# 3. Space Essay (Custom: Emphasize question type)
print('\n' + '─' * 60)
print('📖 Essay 3/3: SOLAR SYSTEM (Question-Type Weighted)')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay space --epochs 50 --topk 5 --question_weight 0.5 --memory_weight 0.3 --dispersion_weight 0.2


## 11. Trainable Decision Maker (MLP Classifier)
This cell demonstrates the **Trainable Decision Maker**:
- Uses an MLP classifier to learn when to use titans_only vs hybrid mode
- Automatic labeling: compares both modes' answer quality
- Training data collected during Q&A, model trained periodically

**Features** (5-dim input):
1. Question Type Score (factual vs reasoning)
2. Memory Confidence (LTM output strength)
3. Retrieval Dispersion (score entropy)
4. Question Length (normalized)
5. Keyword Density

In [ ]:
# Trainable Decision Maker: Training Mode
# Collects data from all 3 essays and trains the MLP classifier

print('=' * 60)
print('TRAINABLE DECISION MAKER - TRAINING')
print('=' * 60)

# Train on Climate Essay (collect samples + train)
print('\n' + '─' * 60)
print('🧠 Training on Climate Essay...')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay climate --epochs 50 --use_trainable_decider --train_decider --train_epochs 100

# Train on AI Essay (load previous model + continue training)
print('\n' + '─' * 60)
print('🧠 Training on AI Essay...')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay ai --epochs 50 --use_trainable_decider --train_decider --train_epochs 100 --decider_model_path decider_model_climate.pt


## 12. Trainable Decision Maker - Inference
Now using the trained MLP classifier for inference on a new essay.

In [ ]:
# Trainable Decision Maker: Inference Mode (WITH fallback disabled)
# Use trained model on Space Essay (unseen during training)
# --disable_fallback: shows pure TrainableDecider output without safety fallback

print('=' * 60)
print('TRAINABLE DECISION MAKER - INFERENCE (Space Essay)')
print('Fallback DISABLED: showing pure decider output')
print('=' * 60)

!python projects/hybrid_titans/hybrid_titan_demo.py --essay space --epochs 50 --use_trainable_decider --decider_model_path decider_model_climate.pt --disable_fallback

print('\\n✅ Trainable Decision Maker Demo Complete!')
